In [4]:
import pandas as pd
import numpy as np
from glob import glob
import re

Arenas played in provided by Sports Reference

In [40]:
cc_shots = pd.read_csv("cc_shots.csv")     
cc_arenas = pd.read_csv("cc_arenas.csv")     

In [41]:
files = sorted(glob("cc_arena_game/cc*.csv"))
print(files)

['cc_arena_game/cc2021.csv', 'cc_arena_game/cc2022.csv', 'cc_arena_game/cc2023.csv', 'cc_arena_game/cc2024.csv', 'cc_arena_game/cc2025.csv']


In [42]:
import pandas as pd, numpy as np, re, io
from glob import glob

def _rename_sr_cols(df):
    cols = list(df.columns)
    # after 'Type' -> HA (home/@/N)
    if "Type" in cols:
        i = cols.index("Type")
        if i+1 < len(cols) and (str(cols[i+1]).startswith("Unnamed") or cols[i+1] in ("", None)):
            df = df.rename(columns={cols[i+1]: "HA"})
    # after 'SRS' -> Result (W/L)
    if "SRS" in cols:
        j = cols.index("SRS")
        if j+1 < len(cols) and (str(cols[j+1]).startswith("Unnamed") or cols[j+1] in ("", None)):
            df = df.rename(columns={cols[j+1]: "Result"})
    return df

def read_sr_schedule_file(path):
    with open(path, "r", encoding="utf-8-sig", errors="ignore") as f:
        lines = f.read().splitlines()

    # find header row (real table start)
    start = next(i for i,l in enumerate(lines)
                 if l.strip().startswith("G,Date,Time,Type") and "Arena" in l)

    # collect table until blank/HTML/"Provided by"
    body = []
    for line in lines[start:]:
        s = line.strip()
        if not s or s.startswith("Provided by") or s.startswith("<"):
            break
        body.append(line)
    if not body:
        raise ValueError(f"{path}: table block empty after trimming preface/footer.")

    df = pd.read_csv(io.StringIO("\n".join(body)), engine="python")
    return _rename_sr_cols(df)

def load_sr_home(files):
    rows = []
    for f in files:
        try:
            season = int(re.search(r"(20\d{2})", f).group(1))
            df = read_sr_schedule_file(f)
            df["HA"] = df.get("HA", "").fillna("").astype(str).str.strip()
            df["is_home"] = ~df["HA"].isin(["@", "N"])  # blank = home
            keep = df[df["is_home"] & df["Arena"].notna()].copy()
            keep["season"] = season
            rows.append(keep[["season", "Opponent", "Arena"]])
        except Exception as e:
            print(f"Skipping {f}: {e}")
    if not rows:
        raise RuntimeError("No valid home rows loaded. Check CSV contents/paths.")
    return pd.concat(rows, ignore_index=True)

# usage
sr_home = load_sr_home(files)
print(sr_home.head())


   season                 Opponent                  Arena
0    2021  Texas A&M-International   American Bank Center
1    2021              Texas State   American Bank Center
2    2021  Texas-Rio Grande Valley   American Bank Center
3    2021     Our Lady of the Lake  Dugan Wellness Center
4    2021               Paul Quinn   American Bank Center


In [43]:
sr_home

,season,Opponent,Arena
0,2021,Texas A&M-International,American Bank Center
1,2021,Texas State,American Bank Center
2,2021,Texas-Rio Grande Valley,American Bank Center
3,2021,Our Lady of the Lake,Dugan Wellness Center
4,2021,Paul Quinn,American Bank Center
...,...,...,...
67,2025,McNeese State,American Bank Center
68,2025,Incarnate Word,American Bank Center
69,2025,Houston Christian,American Bank Center
70,2025,Southeastern Louisiana,American Bank Center


In [44]:
cc_shots

,gameId,homeMarket,awayMarket,secsIntoGame,teamMarket,actionType,subType,success,side,shotDist,zones6,zones13,season,team,Year,conf,similar_team,half,primary_side,side_mismatch
0,1984957,A&M-Corpus Christi,Northwestern St.,124.0,Northwestern St.,2pt,jumpshot,False,LEFT,20.004,mid2,rb2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False
1,1984957,A&M-Corpus Christi,Northwestern St.,352.0,Northwestern St.,2pt,jumpshot,False,LEFT,15.822,mid2,le2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False
2,1984957,A&M-Corpus Christi,Northwestern St.,520.0,Northwestern St.,3pt,jumpshot,True,LEFT,23.411,atb3,lw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False
3,1984957,A&M-Corpus Christi,Northwestern St.,585.0,Northwestern St.,2pt,jumpshot,True,LEFT,17.792,mid2,rb2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False
4,1984957,A&M-Corpus Christi,Northwestern St.,665.0,Northwestern St.,3pt,jumpshot,False,LEFT,24.284,atb3,rw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1674,2184087,A&M-Corpus Christi,CSU Bakersfield,1296.0,CSU Bakersfield,2pt,jumpshot,False,RIGHT,18.312,mid2,re2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,RIGHT,False
1675,2184087,A&M-Corpus Christi,CSU Bakersfield,1620.0,CSU Bakersfield,3pt,stepbackjumpshot,True,RIGHT,23.112,c3,rc3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,RIGHT,False
1676,2184087,A&M-Corpus Christi,CSU Bakersfield,1745.0,CSU Bakersfield,2pt,pullupjumpshot,False,RIGHT,18.950,mid2,le2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,RIGHT,False
1677,2184087,A&M-Corpus Christi,CSU Bakersfield,1851.0,CSU Bakersfield,3pt,jumpshot,False,RIGHT,23.128,c3,rc3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,RIGHT,False


In [57]:
cc_shots

,gameId,homeMarket,awayMarket,secsIntoGame,teamMarket,actionType,subType,success,side,shotDist,zones6,zones13,season,team,Year,conf,similar_team,half,primary_side,side_mismatch
0,1984957,A&M-Corpus Christi,Northwestern St.,124.0,Northwestern St.,2pt,jumpshot,False,LEFT,20.004,mid2,rb2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False
1,1984957,A&M-Corpus Christi,Northwestern St.,352.0,Northwestern St.,2pt,jumpshot,False,LEFT,15.822,mid2,le2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False
2,1984957,A&M-Corpus Christi,Northwestern St.,520.0,Northwestern St.,3pt,jumpshot,True,LEFT,23.411,atb3,lw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False
3,1984957,A&M-Corpus Christi,Northwestern St.,585.0,Northwestern St.,2pt,jumpshot,True,LEFT,17.792,mid2,rb2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False
4,1984957,A&M-Corpus Christi,Northwestern St.,665.0,Northwestern St.,3pt,jumpshot,False,LEFT,24.284,atb3,rw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1674,2184087,A&M-Corpus Christi,CSU Bakersfield,1296.0,CSU Bakersfield,2pt,jumpshot,False,RIGHT,18.312,mid2,re2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,RIGHT,False
1675,2184087,A&M-Corpus Christi,CSU Bakersfield,1620.0,CSU Bakersfield,3pt,stepbackjumpshot,True,RIGHT,23.112,c3,rc3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,RIGHT,False
1676,2184087,A&M-Corpus Christi,CSU Bakersfield,1745.0,CSU Bakersfield,2pt,pullupjumpshot,False,RIGHT,18.950,mid2,le2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,RIGHT,False
1677,2184087,A&M-Corpus Christi,CSU Bakersfield,1851.0,CSU Bakersfield,3pt,jumpshot,False,RIGHT,23.128,c3,rc3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,RIGHT,False


In [58]:
cc_shots = cc_shots[cc_shots["awayMarket"]!="CSU Bakersfield"]

In [59]:
sr_map["Opponent"].unique()

array(['Texas A&M-International', 'Texas State', 'UTRGV',
       'Our Lady of the Lake', 'Paul Quinn', 'Sam Houston', 'SFA', 'UIW',
       'Abilene Christian', 'Lamar University', 'Southeastern La.',
       'New Orleans', 'Texas Lutheran', "St. Mary's (TX)",
       'Southwestern (TX)', 'Sul Ross State', 'McNeese',
       'Houston Christian', 'Northwestern St.', 'Nicholls', 'UTSA',
       'Trinity (TX)', 'Schreiner', 'East Texas A&M',
       'Southwestern Adventist', 'Dallas Christian', 'Omaha',
       'Tennessee-Martin', 'Le Moyne', 'Prairie View'], dtype=object)

In [60]:
opp_map = {
    "Stephen F. Austin":"SFA",
    "Incarnate Word":"UIW",
    "Southeastern Louisiana":"Southeastern La.",
    "Northwestern State":"Northwestern St.",
    "McNeese State":"McNeese",
    "Texas-Rio Grande Valley":"UTRGV",
    "Nicholls State":"Nicholls",
    "Lamar":"Lamar University",
}

sr_home = sr_home.copy()
sr_home["Opponent"] = sr_home["Opponent"].replace(opp_map)

# ensure 1 row per (season, Opponent)
sr_map = sr_home[["season","Opponent","Arena"]].drop_duplicates(["season","Opponent"])

shots_with_arena = (
    cc_shots.merge(sr_map, left_on=["season","awayMarket"], right_on=["season","Opponent"], how="left")
            .drop(columns=["Opponent"])
)

shots_with_arena["Arena"].isna().sum()

np.int64(0)

In [61]:
shots_with_arena

,gameId,homeMarket,awayMarket,secsIntoGame,teamMarket,actionType,subType,success,side,shotDist,...,zones13,season,team,Year,conf,similar_team,half,primary_side,side_mismatch,Arena
0,1984957,A&M-Corpus Christi,Northwestern St.,124.0,Northwestern St.,2pt,jumpshot,False,LEFT,20.004,...,rb2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False,American Bank Center
1,1984957,A&M-Corpus Christi,Northwestern St.,352.0,Northwestern St.,2pt,jumpshot,False,LEFT,15.822,...,le2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False,American Bank Center
2,1984957,A&M-Corpus Christi,Northwestern St.,520.0,Northwestern St.,3pt,jumpshot,True,LEFT,23.411,...,lw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False,American Bank Center
3,1984957,A&M-Corpus Christi,Northwestern St.,585.0,Northwestern St.,2pt,jumpshot,True,LEFT,17.792,...,rb2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False,American Bank Center
4,1984957,A&M-Corpus Christi,Northwestern St.,665.0,Northwestern St.,3pt,jumpshot,False,LEFT,24.284,...,rw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,LEFT,False,American Bank Center
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1629,1987734,A&M-Corpus Christi,Houston Christian,1906.0,Houston Christian,3pt,jumpshot,False,RIGHT,25.703,...,lw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,2,RIGHT,False,American Bank Center
1630,2190799,A&M-Corpus Christi,New Orleans,137.0,New Orleans,2pt,jumpshot,True,LEFT,17.317,...,re2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,1,LEFT,False,American Bank Center
1631,2190799,A&M-Corpus Christi,New Orleans,1197.8,New Orleans,3pt,jumpshot,False,LEFT,23.371,...,lc3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,1,LEFT,False,American Bank Center
1632,2190799,A&M-Corpus Christi,New Orleans,1410.0,New Orleans,2pt,jumpshot,True,RIGHT,18.387,...,re2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,RIGHT,False,American Bank Center


In [63]:
cc_arenas

,School,Conference,Wall Number,Side,Distance,Type of Wall,Arena,Left_Wall,Right_Wall,Left_Wall_Distance,Right_Wall_Distance,Left_Wall_Type,Right_Wall_Type
0,Texas A&M CC 1,Southland,2,Both,Close,Whole,Dugan Wellness Center,True,True,Close,Close,Whole,Whole
1,Texas A&M-Corpus Christi,Southland,1,Left,Far,Mix,American Bank Center Arena,True,False,Far,NaN,Mix,NaN


In [64]:
shots_with_arena.merge(cc_arenas,on="Arena")

,gameId,homeMarket,awayMarket,secsIntoGame,teamMarket,actionType,subType,success,side,shotDist,...,Wall Number,Side,Distance,Type of Wall,Left_Wall,Right_Wall,Left_Wall_Distance,Right_Wall_Distance,Left_Wall_Type,Right_Wall_Type
0,2189338,A&M-Corpus Christi,UIW,247.0,UIW,3pt,jumpshot,False,LEFT,25.100,...,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
1,2189338,A&M-Corpus Christi,UIW,596.0,UIW,2pt,jumpshot,True,LEFT,20.256,...,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
2,2189338,A&M-Corpus Christi,UIW,678.0,UIW,3pt,jumpshot,False,LEFT,24.419,...,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
3,2189338,A&M-Corpus Christi,UIW,1364.0,UIW,2pt,jumpshot,True,RIGHT,19.602,...,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
4,2189338,A&M-Corpus Christi,UIW,1402.0,UIW,2pt,jumpshot,False,RIGHT,15.788,...,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,2551171,A&M-Corpus Christi,Lamar University,1311.0,Lamar University,2pt,jumpshot,True,RIGHT,15.563,...,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
65,2551171,A&M-Corpus Christi,Lamar University,1763.0,Lamar University,2pt,jumpshot,False,RIGHT,15.829,...,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
66,2551171,A&M-Corpus Christi,Lamar University,1853.0,Lamar University,3pt,jumpshot,True,RIGHT,25.053,...,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
67,2551171,A&M-Corpus Christi,Lamar University,2197.0,Lamar University,3pt,jumpshot,False,RIGHT,24.363,...,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
